# 2D loss landscapes

2D version of `1104_loss_landscape_1d.ipynb`: around one reference configuration, sweep every
pair of `tongueIndex`, `tongueDiameter`, `lipDiameter` and `voiceness` on a grid (everything else held at the
reference values) and show the MFCC loss against the reference as a heatmap. The cross marks
the reference.

The grid is the model's 32 bucket centers per parameter, i.e. exactly the values the controller
can output at eval time.

The loss is computed on the middle MFCC frame of a short (93 ms) clip rather than averaged over
1 s as in training. Parameters are constant over time, so the two agree: on this config the
correlation between the 1-frame and full landscapes was 0.999, at ~8x less compute.

Same setup as training: `pink_trombone_ola` at 512 samples per frame with 256-sample impulse
responses, `MFCCLoss(n_fft=2048)`, and the same noise seed for the reference and every candidate.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import itertools

import plotly.graph_objects as go
import torch
from IPython.display import Audio

from samuel.losses import MFCCLoss
from samuel.model import _DEFAULT_FROZEN_VALUES, _DEFAULT_PARAM_SPEC, PinkTromboneControllerConfig
from samuel.pink_trombone import PARAM_NAMES, SAMPLE_RATE, pink_trombone_ola

SAMPLES_PER_FRAME = 512
CONTROL_RATE = SAMPLE_RATE / SAMPLES_PER_FRAME
# 4096 samples -> 5 MFCC frames (n_fft 2048, hop 512); the middle one covers samples 1024-3072.
N_FRAMES = 8
SEED = 0
N_BUCKETS = PinkTromboneControllerConfig().n_buckets  # 32
DEVICE = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

# The loss stays on CPU: torch.stft on MPS crashed the kernel. It's cheap next to the synth.
loss_fn = MFCCLoss(samples_per_frame=SAMPLES_PER_FRAME, n_fft=2048)

## Reference configuration

Config 7 from `1104_loss_landscape_1d.ipynb`, with `tongueDiameter` lowered from 3.49 to 3.

In [ ]:
config = {
    **_DEFAULT_FROZEN_VALUES,
    "voiceness": 0.64,
    "intensity": 0.74,
    "tongueIndex": 30.49,
    "tongueDiameter": 3.0,
    "constrictionIndex": 33.87,
    "constrictionDiameter": 2.01,
    "lipDiameter": 2.62,
    "frequency": 108.78,
}

In [ ]:
def make_params(overrides: dict[str, torch.Tensor], n_frames: int = N_FRAMES) -> torch.Tensor:
    """``overrides`` maps param name -> [B] values; returns [B, n_frames, 12], constant over time."""
    B = len(next(iter(overrides.values())))
    base = torch.tensor([config[n] for n in PARAM_NAMES], device=DEVICE)
    params = base.expand(B, n_frames, -1).clone()
    for name, values in overrides.items():
        params[..., PARAM_NAMES.index(name)] = values.to(DEVICE)[:, None]
    return params


def synth(params: torch.Tensor) -> torch.Tensor:
    # A [B] tensor of equal seeds gives every row the same noise; a scalar seed
    # would draw a different noise row per batch slot.
    seed = torch.full((len(params),), SEED)
    return pink_trombone_ola(params, seed=seed, control_rate=CONTROL_RATE).cpu()


def middle_frame_features(audio: torch.Tensor) -> torch.Tensor:
    """[B, S] -> [B, n_mfcc] MFCCs of the middle frame."""
    feats = loss_fn.features(audio)  # [B, n_mfcc, T]
    return feats[..., feats.shape[-1] // 2]


ref_overrides = {"tongueIndex": torch.tensor([config["tongueIndex"]])}
with torch.no_grad():
    ref_features = middle_frame_features(synth(make_params(ref_overrides)))
    # 1 s for listening only.
    ref_audio = synth(make_params(ref_overrides, n_frames=round(CONTROL_RATE)))
Audio(ref_audio[0].numpy(), rate=SAMPLE_RATE)

## Sweep

In [ ]:
SWEPT = ["tongueIndex", "tongueDiameter", "lipDiameter", "voiceness"]
PAIRS = list(itertools.combinations(SWEPT, 2))


def landscape(x_name: str, y_name: str, batch_size: int = 64):
    """Loss over the bucket-center grid -> (xs, ys, loss [N_BUCKETS, N_BUCKETS])."""
    xs = torch.linspace(*_DEFAULT_PARAM_SPEC[x_name][:2], N_BUCKETS)
    ys = torch.linspace(*_DEFAULT_PARAM_SPEC[y_name][:2], N_BUCKETS)
    yy, xx = torch.meshgrid(ys, xs, indexing="ij")
    losses = []
    with torch.no_grad():
        for xc, yc in zip(xx.flatten().split(batch_size), yy.flatten().split(batch_size)):
            feats = middle_frame_features(synth(make_params({x_name: xc, y_name: yc})))
            losses.append((feats - ref_features).abs().mean(dim=-1))
    return xs, ys, torch.cat(losses).view(N_BUCKETS, N_BUCKETS)

## Results

Each pair is shown as soon as it's computed.

In [ ]:
def plot_landscape(x_name, y_name, xs, ys, loss):
    fig = go.Figure(
        go.Heatmap(
            x=xs, y=ys, z=loss, colorscale="Viridis", colorbar_title="MFCC loss",
            hovertemplate=f"{x_name}=%{{x:.2f}}<br>{y_name}=%{{y:.2f}}<br>loss=%{{z:.3f}}<extra></extra>",
        )
    )
    fig.add_trace(
        go.Scatter(
            x=[config[x_name]], y=[config[y_name]], mode="markers", showlegend=False,
            marker=dict(symbol="x", size=12, color="white", line=dict(width=1, color="black")),
        )
    )
    fig.update_layout(
        width=520, height=440, margin=dict(t=50, b=40),
        title=f"{x_name} × {y_name}", xaxis_title=x_name, yaxis_title=y_name,
    )
    fig.show()


for x_name, y_name in PAIRS:
    plot_landscape(x_name, y_name, *landscape(x_name, y_name))